In [ ]:
import os, glob, random, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models

# ------------------------- CONFIG ------------------------------------
ZIP_PATH = None            
RAW_DIR = "/content/raw"   
DATA_DIR = "/content/data" 
MAX_PER_CLASS = 5000       
VAL_SPLIT = 0.2
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42
random.seed(SEED)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

# ------------------------- 1. UNZIP ----------------------------------
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)    # start clean on every run

if ZIP_PATH is None:
    zips = [z for z in glob.glob("/content/*.zip")]
    assert zips, "No .zip found in /content. Upload your zip or set ZIP_PATH."
    ZIP_PATH = zips[0]
print("Using zip:", ZIP_PATH)

os.makedirs(RAW_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(RAW_DIR)

# Some Kaggle downloads contain zips inside the zip (e.g. train.zip) - extract them too
for inner in glob.glob(os.path.join(RAW_DIR, "**", "*.zip"), recursive=True):
    out = os.path.splitext(inner)[0]
    with zipfile.ZipFile(inner) as zf:
        zf.extractall(out)
    print("Extracted inner zip:", inner)

# ------------------------- 2. FIND + LABEL IMAGES ---------------------
def get_label(path):
    """Return 'cats', 'dogs' or None, using folder names first, then file name."""
    rel = os.path.relpath(path, RAW_DIR).lower()
    parts = rel.split(os.sep)
    if any("test" in p for p in parts[:-1]):
        return None                                  # skip unlabeled test images
    for p in reversed(parts[:-1]):                   # folder names
        if "cat" in p:
            return "cats"
        if "dog" in p:
            return "dogs"
    name = parts[-1]                                 # file name (cat.0.jpg, dog.5.jpg)
    if name.startswith("cat"):
        return "cats"
    if name.startswith("dog"):
        return "dogs"
    return None

def is_valid_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

files = {"cats": [], "dogs": []}
for p in glob.glob(os.path.join(RAW_DIR, "**", "*"), recursive=True):
    if p.lower().endswith(IMG_EXT):
        label = get_label(p)
        if label:
            files[label].append(p)

print({k: len(v) for k, v in files.items()}, "images found")
assert files["cats"] and files["dogs"], (
    "Could not find labeled cat/dog images. Run: !find /content/raw | head -30 "
    "and check the folder/file names."
)

# ------------------------- 3. CLEAN + SPLIT --------------------------
for label in files:
    random.shuffle(files[label])
    good = []
    for p in files[label]:                           # skip corrupt images
        if is_valid_image(p):
            good.append(p)
        if MAX_PER_CLASS and len(good) >= MAX_PER_CLASS:
            break
    n_val = int(len(good) * VAL_SPLIT)
    for split, subset in [("val", good[:n_val]), ("train", good[n_val:])]:
        out = os.path.join(DATA_DIR, split, label)
        os.makedirs(out, exist_ok=True)
        for i, p in enumerate(subset):
            ext = os.path.splitext(p)[1]
            shutil.copy(p, os.path.join(out, f"{label[:-1]}_{i}{ext}"))
    print(f"{label}: {len(good) - n_val} train, {n_val} val")

# ------------------------- 4. LOAD DATASETS --------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"), image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="binary", seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "val"), image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="binary", shuffle=False,
)
class_names = train_ds.class_names                   # ['cats', 'dogs'] -> cats=0, dogs=1
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# ------------------------- 5. MODEL (3 CONV LAYERS) -------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, (3, 3), activation="relu"),    # Conv layer 1
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),    # Conv layer 2
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),   # Conv layer 3
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),           # probability of "dog"
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# ------------------------- 6. TRAIN ----------------------------------
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                     restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("dog_cat_cnn.keras", save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS, callbacks=callbacks)

# ------------------------- 7. EVALUATE -------------------------------
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"\nValidation loss: {loss:.4f} | Validation accuracy: {acc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in val_ds]).ravel().astype(int)
y_pred = (model.predict(val_ds, verbose=0) > 0.5).astype(int).ravel()
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

# ------------------------- 8. PLOTS ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig("dog_cat_training.png", dpi=150)
plt.show()

# Show some validation predictions
plt.figure(figsize=(12, 6))
images, labels = next(iter(val_ds.unbatch().shuffle(500).batch(12)))
probs = model.predict(images, verbose=0).ravel()
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    pred = class_names[int(probs[i] > 0.5)][:-1]
    true = class_names[int(labels[i][0])][:-1]
    plt.title(f"pred: {pred}\ntrue: {true}", color="green" if pred == true else "red")
    plt.axis("off")
plt.tight_layout()
plt.savefig("dog_cat_predictions.png", dpi=150)
plt.show()

# ------------------------- 9. PREDICT YOUR OWN IMAGE ------------------
def predict_image(path):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[np.newaxis, ...]
    p = float(model.predict(arr, verbose=0)[0][0])
    label = class_names[1] if p > 0.5 else class_names[0]
    conf = p if p > 0.5 else 1 - p
    print(f"{path}: {label[:-1]} ({conf:.1%} confident)")
